# Optimizer Internals

Everyone uses AdamW and almost nobody can say what it does differently from Adam, or why
warmup is needed, or why the same learning rate that worked at one model size fails at
another. Those are not trivia — they are the three most common causes of a training run
that underperforms for no visible reason.

This notebook implements SGD, momentum, Adam and AdamW from scratch on problems where the
right answer is known, so each mechanism can be seen doing its job.

Final topic in the [RL & Training Dynamics](training-dynamics.ipynb) track.

## 1. What & Why

The problem every optimiser is solving: **loss landscapes are badly conditioned**. Some
directions are steep and some are nearly flat, and plain gradient descent must use a step
size small enough for the steepest direction — which means it crawls along the flat ones.

The condition number `κ` (ratio of largest to smallest curvature) determines how bad this
is, and for neural networks it is enormous. The responses:

- **Momentum** accumulates a velocity, so consistent progress along a flat direction
  compounds while oscillation across a steep one cancels.
- **Adaptive methods (Adam)** keep a per-parameter estimate of gradient magnitude and
  divide by it, so every parameter gets a step of comparable *relative* size regardless of
  its gradient scale.
- **Warmup** exists because those estimates start empty and are meaningless for the first
  few hundred steps.
- **Learning-rate schedules** exploit the fact that a large step size is right early
  (when you are far away) and wrong later (when you need to settle).

**Why AdamW rather than Adam.** L2 regularisation added to the gradient gets divided by
Adam's per-parameter scaling along with everything else — so parameters with large
gradients are decayed *less*, which is the opposite of what regularisation should do.
AdamW applies the decay directly to the weights, outside the adaptive scaling. This is a
one-line change and it measurably improves generalisation.

## 2. Mental Model

**A ball rolling down a long, narrow valley.**

The valley floor descends gently toward the minimum; the walls are steep. This is what
"ill-conditioned" looks like, and it is the normal case.

- **Plain SGD** must take steps small enough not to fly up the walls, so it inches along
  the floor. Increase the step size and it oscillates between the walls, making no
  progress along the valley.
- **Momentum** is the ball having mass. Side-to-side wall bounces alternate in direction
  and cancel; downhill motion along the floor is consistent and accumulates. The ball
  gains speed in exactly the direction you want.
- **Adam** rescales each axis by how large its gradients have been, which *reshapes the
  valley into a bowl*. The steep wall direction is divided by a large number, the flat
  floor direction by a small one, and a single step size becomes appropriate for both.

That reshaping is why Adam is so robust to learning-rate choice, and it is also why its
step size means something different from SGD's: an Adam step is roughly `lr` in *relative*
terms per parameter, regardless of the gradient's magnitude. A well-behaved run keeps
`‖Δw‖/‖w‖` near `1e-3` — the diagnostic from
[Training Dynamics](training-dynamics.ipynb).

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Condition number `κ`** | Ratio of largest to smallest curvature. SGD's convergence rate degrades roughly linearly in `κ`. |
| **Momentum `β₁`** | Exponential moving average of the gradient. Typically 0.9. |
| **Second moment `β₂`** | EMA of the *squared* gradient — Adam's per-parameter scale estimate. Typically 0.999. |
| **Bias correction** | Dividing by `1 − βᵗ`. Without it the EMAs are biased toward their zero initialisation for the first `~1/(1−β)` steps. |
| **`ε`** | Added to the denominator for numerical safety. Larger `ε` makes Adam behave more like SGD. |
| **Decoupled weight decay** | AdamW: apply decay to the weights directly, not through the gradient and adaptive scaling. |
| **Warmup** | Ramping the LR from ~0 so the second-moment estimate can stabilise first. |
| **Cosine schedule** | Smooth LR decay to near zero. The current default for LLM pre-training. |
| **Effective LR** | `lr / (√v + ε)` for Adam — what actually multiplies the gradient, per parameter. |
| **Muon / second-order** | Newer optimisers using orthogonalised or curvature-aware updates for matrix-shaped parameters. |

## 4. Setup

NumPy. Optimisers are compared on an ill-conditioned quadratic, where the optimum is known
exactly and convergence is unambiguous — no model quality confound.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — ill-conditioning, and what each optimiser does about it

A quadratic `f(w) = ½ wᵀ A w` with a controllable condition number. The optimum is `w = 0`,
so distance from it measures convergence exactly.

In [2]:
def make_quadratic(dim=50, kappa=1000.0, seed=0):
    '''Diagonal curvature spanning [1, kappa] -- a long narrow valley in `dim` axes.'''
    return np.logspace(0, np.log10(kappa), dim)

def run(opt, curv, lr, steps=500, w0=None, b1=0.9, b2=0.999, eps=1e-8, wd=0.0):
    w = (np.ones(len(curv)) if w0 is None else w0.copy())
    m = np.zeros_like(w); v = np.zeros_like(w); vel = np.zeros_like(w)
    dist = []
    for t in range(1, steps + 1):
        g = curv * w                              # gradient of 0.5 * w^T A w
        if opt == "sgd":
            w = w - lr * g
        elif opt == "momentum":
            vel = b1 * vel + g
            w = w - lr * vel
        elif opt in ("adam", "adamw"):
            m = b1 * m + (1 - b1) * g
            v = b2 * v + (1 - b2) * g ** 2
            mh, vh = m / (1 - b1 ** t), v / (1 - b2 ** t)
            if opt == "adam":
                w = w - lr * (mh / (np.sqrt(vh) + eps) + wd * w)      # L2 in the gradient
            else:
                w = w - lr * mh / (np.sqrt(vh) + eps) - lr * wd * w   # decoupled
        dist.append(float(np.linalg.norm(w)))
    return np.array(dist)

curv = make_quadratic(kappa=1000.0)
lr_sgd = 1.0 / curv.max()          # the largest stable step for SGD

print(f"condition number: {curv.max()/curv.min():.0f}")
print(f"largest stable SGD learning rate: {lr_sgd:.2e}\n")
print(f"{'optimiser':12} {'lr':>10} {'dist @100':>11} {'dist @500':>11}")
configs = [("sgd", lr_sgd), ("momentum", lr_sgd), ("adam", 0.02), ("adam", 0.05)]
for opt, lr in configs:
    d = run(opt, curv, lr)
    print(f"{opt:12} {lr:10.2e} {d[99]:11.4f} {d[-1]:11.4f}")

print("\nSGD is limited by the STEEPEST direction and therefore crawls along the")
print("flattest one -- it has barely moved after 500 steps. Momentum, at the same")
print("learning rate, does substantially better by accumulating consistent progress.")
print("\nAdam does better still, and note its learning rate is orders of magnitude")
print("larger. That is not a fairer setting sneaked in -- it is the point: dividing by")
print("the per-parameter gradient scale makes the step size scale-free, so a single lr")
print("is appropriate for every axis at once.")

condition number: 1000
largest stable SGD learning rate: 1.00e-03

optimiser            lr   dist @100   dist @500
sgd            1.00e-03      3.0111      1.3198
momentum       1.00e-03      0.6095      0.0045
adam           2.00e-02      0.0552      0.0000
adam           5.00e-02      0.0298      0.0000

SGD is limited by the STEEPEST direction and therefore crawls along the
flattest one -- it has barely moved after 500 steps. Momentum, at the same
learning rate, does substantially better by accumulating consistent progress.

Adam does better still, and note its learning rate is orders of magnitude
larger. That is not a fairer setting sneaked in -- it is the point: dividing by
the per-parameter gradient scale makes the step size scale-free, so a single lr
is appropriate for every axis at once.


### Example 2 — bias correction is not optional

Adam's moments start at zero, so early estimates are biased toward zero. The correction
factor `1/(1−βᵗ)` undoes exactly that. Removing it does something specific and bad.

In [3]:
g_const = 0.1        # a constant gradient, so the true EMA target is exactly 0.1
b1, b2 = 0.9, 0.999
m = v = 0.0
print(f"a constant gradient of {g_const}; the EMAs should converge to it\n")
print(f"{'step':>6} {'m (raw)':>10} {'m corrected':>13} | {'sqrt(v) raw':>13} "
      f"{'sqrt(v) corr':>13} | {'step size ratio':>16}")
for t in range(1, 1002):
    m = b1 * m + (1 - b1) * g_const
    v = b2 * v + (1 - b2) * g_const ** 2
    mh, vh = m / (1 - b1 ** t), v / (1 - b2 ** t)
    if t in (1, 2, 10, 100, 1000):
        raw_step = m / (np.sqrt(v) + 1e-8)
        cor_step = mh / (np.sqrt(vh) + 1e-8)
        print(f"{t:6d} {m:10.5f} {mh:13.5f} | {np.sqrt(v):13.5f} {np.sqrt(vh):13.5f} "
              f"| {raw_step/cor_step:16.3f}")

print("\nBoth raw EMAs start far below their true value, but they recover at DIFFERENT")
print("rates -- m with beta1=0.9 recovers in tens of steps, v with beta2=0.999 in")
print("thousands. Because Adam divides one by the square root of the other, the")
print("mismatch does not cancel.")
print("\nThe last column is the practical consequence. Without bias correction the first")
print("update is over 3x too large, it gets WORSE before it gets better (peaking above")
print("6x around step 10, as m recovers long before v does), and it is still noticeably")
print("off after a thousand steps.")
print("\nThat is a large, systematic step-size error at exactly the moment the model is")
print("most fragile -- and note that the correction does not fully rescue the early")
print("steps either, because a corrected estimate built from two samples is still an")
print("estimate from two samples. That residual is part of why warmup helps.")

a constant gradient of 0.1; the EMAs should converge to it

  step    m (raw)   m corrected |   sqrt(v) raw  sqrt(v) corr |  step size ratio
     1    0.01000       0.10000 |       0.00316       0.10000 |            3.162
     2    0.01900       0.10000 |       0.00447       0.10000 |            4.250
    10    0.06513       0.10000 |       0.00998       0.10000 |            6.528
   100    0.10000       0.10000 |       0.03086       0.10000 |            3.241
  1000    0.10000       0.10000 |       0.07952       0.10000 |            1.258

Both raw EMAs start far below their true value, but they recover at DIFFERENT
rates -- m with beta1=0.9 recovers in tens of steps, v with beta2=0.999 in
thousands. Because Adam divides one by the square root of the other, the
mismatch does not cancel.

The last column is the practical consequence. Without bias correction the first
update is over 3x too large, it gets WORSE before it gets better (peaking above
6x around step 10, as m recovers long be

### Example 3 — Adam's L2 is not AdamW's weight decay

The distinction that gives AdamW its name. With adaptive scaling, L2-in-the-gradient
decays each parameter by an amount that depends on its gradient history — which is not
what regularisation is supposed to do.

In [4]:
b1, b2, eps, LR = 0.9, 0.999, 1e-8, 0.01

def final_weight(mode, grad_scale, wd, steps=200):
    w, m, v = 1.0, 0.0, 0.0
    for t in range(1, steps + 1):
        g = grad_scale * 0.5                    # a steady gradient at this param's scale
        if mode == "adam_l2":
            g = g + wd * w                      # L2 folded into the gradient
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        w -= LR * (m / (1 - b1**t)) / (np.sqrt(v / (1 - b2**t)) + eps)
        if mode == "adamw":
            w -= LR * wd * w                    # decoupled: straight onto the weight
    return w

# Isolate the DECAY by differencing against the same run with wd = 0. Otherwise the
# gradient step dominates and hides the effect entirely.
print("how much decay each parameter actually receives (wd = 0.1 vs wd = 0):\n")
print(f"{'parameter':24} {'Adam + L2':>12} {'AdamW':>10}")
for name, gs in [("large-gradient param", 1.0), ("small-gradient param", 0.01)]:
    d_adam = final_weight("adam_l2", gs, 0.0) - final_weight("adam_l2", gs, 0.1)
    d_adamw = final_weight("adamw", gs, 0.0) - final_weight("adamw", gs, 0.1)
    print(f"{name:24} {d_adam:12.4f} {d_adamw:10.4f}")

print("\nUnder AdamW both parameters receive the SAME multiplicative decay, because it")
print("is applied to the weight directly: that is what 'decoupled' means, and it is")
print("what makes weight decay behave like regularisation.")
print("\nUnder Adam+L2 the decay term is pushed through the same 1/sqrt(v) scaling as")
print("the gradient. A parameter with large gradients has a large v, so its L2 term is")
print("divided down and it is regularised LESS -- exactly backwards from the intent.")
print("\nThis is why `weight_decay` in Adam and in AdamW are not the same hyperparameter")
print("and do not transfer between them.")

how much decay each parameter actually receives (wd = 0.1 vs wd = 0):

parameter                   Adam + L2      AdamW
large-gradient param          -0.1459    -0.0070
small-gradient param          -0.9737    -0.0070

Under AdamW both parameters receive the SAME multiplicative decay, because it
is applied to the weight directly: that is what 'decoupled' means, and it is
what makes weight decay behave like regularisation.

Under Adam+L2 the decay term is pushed through the same 1/sqrt(v) scaling as
the gradient. A parameter with large gradients has a large v, so its L2 term is
divided down and it is regularised LESS -- exactly backwards from the intent.

This is why `weight_decay` in Adam and in AdamW are not the same hyperparameter
and do not transfer between them.


### Example 4 — schedules: warmup and cosine decay

Two mechanisms, doing different jobs. Warmup protects the start; decay settles the end.

In [5]:
def schedule(step, total, kind, base_lr=0.05, warmup=200):
    if kind == "constant":
        return base_lr
    if kind == "cosine":
        return base_lr * 0.5 * (1 + np.cos(np.pi * step / total))
    if kind == "warmup+cosine":
        if step < warmup:
            return base_lr * (step + 1) / warmup
        p = (step - warmup) / max(1, total - warmup)
        return base_lr * 0.5 * (1 + np.cos(np.pi * p))
    raise ValueError(kind)

def run_scheduled(curv, kind, steps=2000, base_lr=0.05, noise=0.4, seed=0):
    r = np.random.default_rng(seed)
    w = np.ones(len(curv))
    m = np.zeros_like(w); v = np.zeros_like(w)
    b1, b2 = 0.9, 0.999
    for t in range(1, steps + 1):
        g = curv * w + r.normal(0, noise, len(w))     # stochastic gradient
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        lr = schedule(t - 1, steps, kind, base_lr)
        w -= lr * (m / (1 - b1**t)) / (np.sqrt(v / (1 - b2**t)) + 1e-8)
    return float(0.5 * np.sum(curv * w ** 2))

print("final loss on a NOISY ill-conditioned quadratic (lower is better):\n")
print(f"{'schedule':18} " + " ".join(f"{'lr=' + str(lr):>12}" for lr in (0.02, 0.05, 0.1)))
for kind in ("constant", "cosine", "warmup+cosine"):
    row = " ".join(f"{run_scheduled(curv, kind, base_lr=lr):12.5f}"
                   for lr in (0.02, 0.05, 0.1))
    print(f"{kind:18} " + row)

print("\nWith gradient noise, a constant learning rate cannot settle -- it keeps taking")
print("steps of fixed size and bounces around the optimum forever. Decaying the rate")
print("lets the later steps average that noise away, which is most of why every LLM")
print("pre-training run uses a decay schedule.")
print("\nThe warmup variant is not dramatically better here, and it should not be: this")
print("problem is convex and has no sharp early curvature to fall off. Its value shows")
print("up in the settings of Example 2 and in")
print("[Training Dynamics](training-dynamics.ipynb) -- protecting the first few hundred")
print("steps, when Adam's second-moment estimate is still meaningless.")

final loss on a NOISY ill-conditioned quadratic (lower is better):

schedule                lr=0.02      lr=0.05       lr=0.1
constant                0.04015      0.11309      0.22998
cosine                  0.00107      0.00125      0.00136
warmup+cosine           0.00102      0.00110      0.00127

With gradient noise, a constant learning rate cannot settle -- it keeps taking
steps of fixed size and bounces around the optimum forever. Decaying the rate
lets the later steps average that noise away, which is most of why every LLM
pre-training run uses a decay schedule.

The warmup variant is not dramatically better here, and it should not be: this
problem is convex and has no sharp early curvature to fall off. Its value shows
up in the settings of Example 2 and in
[Training Dynamics](training-dynamics.ipynb) -- protecting the first few hundred
steps, when Adam's second-moment estimate is still meaningless.


## 6. Gotchas & Pitfalls

- **Transferring `weight_decay` between Adam and AdamW.** Example 3. They are different
  quantities; AdamW typically wants a larger value.
- **No warmup with Adam.** The second moment needs samples before it is meaningful, and
  bias correction only partly compensates (Example 2).
- **Assuming a learning rate transfers across model sizes.** It does not. The optimal LR
  scales with width and depth; use a principled parameterisation (µP) or re-tune.
- **Setting `ε` too large.** As `ε` grows, `g/(√v + ε)` tends to `g/ε` and Adam degenerates
  into SGD. This is occasionally deliberate; more often it is an accident.
- **Weight-decaying biases and normalisation parameters.** Standard practice is to exclude
  them. Decaying a LayerNorm gain toward zero attacks the network's conditioning.
- **Forgetting that Adam's state is two extra copies of the parameters.** For a large
  model that is a substantial share of memory, which is what 8-bit optimisers and
  sharding target.
- **Comparing optimisers at a single shared learning rate.** Each has a different optimal
  range (Example 1), so a single-LR comparison mostly measures which one happens to suit
  that value.
- **Resuming without the optimiser state.** Restoring the weights but not `m` and `v`
  restarts the moment estimates from zero mid-run — a large, silent perturbation.
- **A cosine schedule with the wrong horizon.** Cosine decay is defined relative to the
  *total* step count. Stopping early leaves the LR high; continuing past the end leaves it
  at zero and training stops.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| Transformers, almost anything modern | **AdamW** + warmup + cosine. The default, and hard to beat |
| Convolutional vision models, large batch | **SGD with momentum** — often generalises slightly better and uses a third of the memory |
| Memory-constrained large-model training | **8-bit Adam**, or **Adafactor** (factored second moment) |
| Matrix-shaped parameters, throughput-sensitive | **Muon** and related orthogonalising optimisers — newer, promising, less battle-tested |
| Very large batch sizes | **LAMB / LARS** — layer-wise normalised updates |
| Fine-tuning | AdamW at a much lower LR, with a short warmup |

**The honest position.** AdamW plus linear warmup plus cosine decay is the default for
good reason, and most alleged improvements do not survive a fair comparison — one where
each optimiser's learning rate is tuned separately, which is exactly the comparison
Example 1 shows is necessary.

What is genuinely worth your attention is not the optimiser but its *interaction* with
scale: learning rate does not transfer across model sizes, and the single most common
cause of a large run underperforming is an LR tuned at small scale and reused. That is
what µP-style parameterisations exist to fix, and it matters more than the choice between
any two optimisers here.

## 8. Resources

- [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980) — Kingma & Ba; Section 3 derives the bias correction of Example 2.
- [Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101) — Loshchilov & Hutter; AdamW, and Example 3's argument in full.
- [On the Convergence of Adam and Beyond](https://arxiv.org/abs/1904.09237) — the known failure cases, and AMSGrad.
- [SGDR: Stochastic Gradient Descent with Warm Restarts](https://arxiv.org/abs/1608.03983) — the origin of the cosine schedule.
- [Tensor Programs V: Tuning Large Neural Networks via Zero-Shot Hyperparameter Transfer](https://arxiv.org/abs/2203.03466) — µP; how to make learning rates transfer across width.
- [Why Momentum Really Works](https://distill.pub/2017/momentum/) — the best visual explanation of the valley picture from the Mental Model section.
- [8-bit Optimizers via Block-wise Quantization](https://arxiv.org/abs/2110.02861) — halving optimiser memory without hurting convergence.
- [Deep Learning Tuning Playbook](https://github.com/google-research/tuning_playbook) — practical guidance on choosing among all of the above.